# Explaining Foundation Models & Ensembles

This example shows how to give the `TimeCopilot` agent your own foundation models and an ensemble, and then ask plain-English questions to understand and compare them.

In [1]:
import nest_asyncio

nest_asyncio.apply()

## Import libraries

In [ ]:
import pandas as pd

from timecopilot import TimeCopilot
from timecopilot.models.foundation.chronos import Chronos
from timecopilot.models.ensembles.median import MedianEnsemble
from timecopilot.models.stats import SeasonalNaive

## Load the dataset

The DataFrame must include at least the following columns:
- unique_id: Unique identifier for each time series (string)
- ds: Date column (datetime format)
- y: Target variable for forecasting (float format)

The pandas frequency will be inferred from the ds column, if not provided.
If the seasonality is not provided, it will be inferred based on the frequency.
If the horizon is not set, it will default to 2 times the inferred seasonality.

In [3]:
df = pd.read_csv("https://timecopilot.s3.amazonaws.com/public/data/air_passengers.csv")
df.head()

,unique_id,ds,y
0,AirPassengers,1949-01-01,112
1,AirPassengers,1949-02-01,118
2,AirPassengers,1949-03-01,132
3,AirPassengers,1949-04-01,129
4,AirPassengers,1949-05-01,121


## Build foundation models and an ensemble

Every model in TimeCopilot is a `Forecaster`, including foundation models and ensembles. Here we instantiate two lightweight, CPU-friendly Chronos checkpoints and a `SeasonalNaive` baseline.

We then wrap them in a `MedianEnsemble`, which combines its members by taking the per-step median of their forecasts, a simple and robust way to blend models.

In [4]:
chronos_bolt = Chronos(repo_id="amazon/chronos-bolt-tiny", alias="Chronos-Bolt")
chronos_t5 = Chronos(repo_id="amazon/chronos-t5-tiny", alias="Chronos-T5")
seasonal_naive = SeasonalNaive()

ensemble = MedianEnsemble(
    models=[chronos_bolt, chronos_t5, seasonal_naive],
    alias="MedianEnsemble",
)

## Initialize the agent with your models

Pass your foundation models and the ensemble to the agent through the `forecasters` parameter. The agent exposes them to the LLM by their `alias`, so it can cross-validate, select, and reason about each one when you ask questions.

In [5]:
tc = TimeCopilot(
    llm="openai:gpt-4o",
    forecasters=[chronos_bolt, chronos_t5, seasonal_naive, ensemble],
    retries=3,
)

## Generate forecast

In [ ]:
result = tc.forecast(df=df)

## Ask questions about the models

Now you can interrogate the analysis in natural language. The agent grounds its answers in the cross-validation results of the models you provided.

In [7]:
print(tc.query("Which model performed best and why?").output)

The model that performed best is the "Chronos-Bolt" model, which achieved the lowest MASE score of 1.2913, as reflected in the evaluation results. The Mean Absolute Scaled Error (MASE) is used to evaluate the performance of forecasting models, and lower scores indicate better performance.

Reasons for Chronos-Bolt's superior performance could include:

1. **Capability to Capture Trends**: The "AirPassengers" time series exhibits a strong trend component, with a trend strength of approximately 0.9972. Chronos-Bolt might be better at capturing trends due to its model architecture or feature selection.

2. **Handling Seasonality**: The series has a marked seasonality with a seasonal period of 12 months. The model likely captures this effectively, indicated by the strong seasonal strength of approximately 0.9815 in the series.

3. **Robustness to Anomalies**: Although anomalies were detected within the data, Chronos-Bolt's performance suggests it effectively handles such irregularities wit

In [8]:
print(
    tc.query(
        "Explain how the MedianEnsemble combines the foundation models' forecasts."
    ).output
)

The "MedianEnsemble" model combines the forecasts from multiple foundational models by taking the median of their predicted values for each time point. This ensemble approach is often used to enhance the stability and accuracy of forecast predictions by leveraging the strengths while minimizing the weaknesses of the individual models.

Here's how the MedianEnsemble likely combines the forecasts:

1. **Aggregate Predictions**: At each time point for which a forecast is needed, the ensemble collects the forecasted values from the different base models used within the ensemble. For instance, models like Chronos-Bolt, Chronos-T5, and SeasonalNaive might all provide predictions for the same month.

2. **Calculate Median**: The median value of these predictions is calculated. The median is a measure of central tendency that is less sensitive to outliers and individual model extremes than the mean. This can be especially beneficial if one or more models produce predictions that are significan